# VN Stock AI — P0.0: Test Qwen2.5-1.5B-Instruct trên Colab T4

Mục đích: chứng minh model gốc **Qwen2.5-1.5B-Instruct** chạy được trên GPU T4 free của Colab, trước khi làm P0 (skill-as-prompt) và P2 (LoRA).

Chạy bằng: Runtime → Change runtime type → T4 GPU → Run all.

Kho Drive dùng chung: `vn-stock-ai/` (models, datasets, checkpoints) — nằm trên Drive TK1, đã share writer cho TK2.

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Cài thư viện (torch đã có sẵn trên Colab)
!pip install -q transformers accelerate

In [ ]:
# 3. Mount Google Drive (kho vn-stock-ai)
from google.colab import drive
drive.mount('/content/drive')

import os
for p in ["/content/drive/MyDrive/vn-stock-ai",
        "/content/drive/Shared with me/vn-stock-ai"]:
    print(p, "->", "CO" if os.path.isdir(p) else "khong thay")
    if os.path.isdir(p):
        print("  ben trong:", os.listdir(p))

In [ ]:
# 4. Load model Qwen2.5-1.5B-Instruct (~3GB bf16, vua T4 16GB)
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)
print("model loaded on:", model.device)

In [ ]:
# 5. Test sinh text: 2 prompt tieng Viet (1 kien thuc chung, 1 nhan dinh co so lieu)
import time

prompts = [
    "Giải thích ngắn gọn trong 2 câu: chỉ số P/E là gì?",
    ("Dữ liệu: mã VCB, giá đóng cửa 95.000, MA20 93.500, RSI(14) 62, "
     "khối ngoại mua ròng 120 tỷ. Nhận định ngắn gọn 2-3 câu về xu hướng ngắn hạn của VCB."),
]

for i, q in enumerate(prompts, 1):
    messages = [
        {"role": "system", "content": "Bạn là trợ lý phân tích chứng khoán Việt Nam. Trả lời bằng tiếng Việt, ngắn gọn."},
        {"role": "user", "content": q},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=256)
    dt = time.time() - t0
    reply = tokenizer.decode(
        out[0][len(inputs.input_ids[0]):], skip_special_tokens=True
    )
    print(f"--- prompt {i} ({dt:.1f}s) ---")
    print(reply.strip())
    print()

## Kết quả kỳ vọng

- GPU T4 nhận, model load xong, 2 prompt đều trả lời tiếng Việt ngắn gọn, mỗi prompt vài giây.
- Nếu prompt 2 nhận định đúng hướng số liệu (giá trên MA20, RSI 62, ngoại mua ròng → xu hướng ngắn hạn tích cực) thì model đủ tốt làm P0.

## Bước tiếp

- P0: đóng gói skill thành system prompt, test trên 20 mã bot đã phân tích.
- P2: LoRA fine-tune trên Qwen2.5-1.5B, checkpoint lưu vào `vn-stock-ai/checkpoints/` trên Drive.